# Strain Gradient Plasticity (2D)

This notebook derives a **higher-order strain gradient plasticity** model following

> P. Gudmundson, *A unified treatment of strain gradient plasticity*, J. Mech. Phys. Solids 52 (2004) 1379-1406.

## Why gradient plasticity?

Conventional ("local") J2 plasticity, as implemented in `src/plasticity.jl`, has **no internal length scale**: the yield stress and hardening depend only on the local accumulated plastic strain. Experiments (micro-indentation, torsion of thin wires, bending of thin foils) show a clear **size effect** (smaller samples are relatively stronger) that local plasticity cannot capture. Gradient plasticity theories introduce a length scale $L$ by making the free energy (and hence the stress-like quantities) depend on **gradients of plastic strain**, not just its local value. This is essential physics at the micron scale (dislocation pile-ups, geometrically necessary dislocations) but invisible at the local model's scale.

`Ratel` does **not** implement this model family, so there is no reference formulation to match here (this derivation follows Gudmundson's paper directly).

## Scope of this notebook

Gudmundson's paper treats the general **tensorial** plastic strain field $\varepsilon^p_{ij}(\mathbf x)$, which requires up to **three independent length scales** and a full higher-order yield/flow theory. We instead implement the paper's own simplified special case (Section 3.1, Eqs. 22-26): a **scalar effective plastic strain field** $\varepsilon^p_e(\mathbf x) \ge 0$, with plastic flow constrained to the conventional $J_2$ (von Mises) direction and a **single length scale** $L$. This:

- directly reuses the isotropic linear + Voce hardening law already implemented for local plasticity (`flow_stress`, `hardening_slope` in `src/plasticity.jl`),
- reduces the extra field from a symmetric tensor (3 components in 2D) to a single scalar per node (much closer in spirit to the mixed (displacement, pressure) formulation in `MixedElasticity2D.ipynb`), and
- is the natural first step; the fully tensorial theory can be revisited later if needed.

## Kinematics: small-strain, not finite-strain

Gudmundson's paper is explicitly a **small-deformation theory** (stated in the paper's introduction). This is a deliberate departure from `src/plasticity.jl`'s finite-strain multiplicative Hencky model ($F = F^e F^p$): here we use the classical **additive** split
$$
\varepsilon_{ij} = \varepsilon^e_{ij} + \varepsilon^p_{ij}, \qquad \varepsilon_{ij} = \tfrac12(u_{i,j}+u_{j,i}),
$$
with the plastic strain rate direction fixed by $J_2$ flow and its magnitude given by the scalar field $\varepsilon^p_e$ (defined below). This is simpler than the finite-strain kinematics (no eigendecomposition, no $F^p$ history tensor) and matches the paper faithfully (the local return-mapping *algebra* (yield function, hardening, Newton solve for $\Delta\gamma$) carries over essentially unchanged from `src/plasticity.jl`, only the strain measure feeding into it changes from a Hencky logarithmic strain to the infinitesimal strain).

As before we work in **plane strain**: $\varepsilon_{33}=\varepsilon^e_{33}+\varepsilon^p_{33}=0$ is enforced implicitly through the 3x3 embedding used for the stress/deviatoric tensor algebra, exactly as in the local model.

## 1. Virtual work and the microforce balance

Gudmundson's framework treats plastic straining as generating its own **microforce balance**, work-conjugate to the plastic strain field, in addition to the ordinary (Cauchy) stress equilibrium. For the scalar simplification, the plastic strain rate is taken colinear with the deviatoric Cauchy stress $\sigma'_{ij}$ (standard $J_2$ flow direction), with the scalar field $\varepsilon^p_e$ giving its magnitude:
$$
\dot\varepsilon^p_{ij} = \dot\varepsilon^p_e \, \frac{3\sigma'_{ij}}{2\sigma_e}, \qquad \sigma_e = \sqrt{\tfrac32 \sigma'_{ij}\sigma'_{ij}} \quad \text{(von Mises equivalent stress)}.
$$
This is exactly the flow direction already used in `src/plasticity.jl`'s return mapping.

With this substitution, the internal virtual work (originally $\int (\sigma_{ij}\delta\varepsilon_{ij} + q_{ij}\delta\varepsilon^p_{ij} + m_{ijk}\delta\varepsilon^p_{ij,k})\,dV$ for the general tensorial theory) collapses to a **scalar microstress** $q$ (conjugate to $\varepsilon^p_e$) and a **moment stress vector** $m_k$ (conjugate to $\varepsilon^p_{e,k}$, i.e. the gradient of the scalar field, no longer a 3rd-order tensor):
$$
\delta w_i = \int_V \Big[ \sigma_{ij}\,\delta\varepsilon_{ij} \;+\; q\,\delta\varepsilon^p_e \;+\; m_k\,\delta\varepsilon^p_{e,k} \Big]\, dV.
$$

**Check of self-consistency.** Substituting the flow rule into $\sigma'_{ij}\delta\varepsilon^p_{ij}$ recovers $q\,\delta\varepsilon^p_e$ with $q=\sigma_e$ exactly at the *local* (non-gradient) level:
$$
\sigma'_{ij}\,\delta\varepsilon^p_{ij} = \sigma'_{ij}\,\delta\varepsilon^p_e\frac{3\sigma'_{ij}}{2\sigma_e} = \delta\varepsilon^p_e\,\frac{3}{2\sigma_e}\sigma'_{ij}\sigma'_{ij} = \delta\varepsilon^p_e\,\frac{3}{2\sigma_e}\cdot\frac{2\sigma_e^2}{3} = \sigma_e\,\delta\varepsilon^p_e,
$$
using $\sigma'_{ij}\sigma'_{ij} = \tfrac23\sigma_e^2$. So in the *absence* of gradient effects $q \to \sigma_e$, recovering ordinary $J_2$ plasticity (the gradient terms are a genuinely new addition, not a reinterpretation of the local physics).

### Equilibrium via the principle of virtual work

Setting $\delta w_i$ equal to the external virtual work $\int_S (t_i\delta u_i + \bar m\,\delta\varepsilon^p_e)\,dS$ for arbitrary admissible $\delta u_i, \delta\varepsilon^p_e$, and integrating $q\,\delta\varepsilon^p_e$ and $\sigma_{ij}\delta\varepsilon_{ij}$ by parts (divergence theorem on the $m_k\delta\varepsilon^p_{e,k}$ term) gives **two coupled field equations**:
$$
\sigma_{ij,j} = 0 \qquad \text{(ordinary stress equilibrium, unchanged)}
$$
$$
m_{k,k} + \sigma_e - q = 0 \qquad \text{(scalar microforce balance)}
$$
with natural (traction-like) boundary conditions $\sigma_{ij}n_j = t_i$ and $m_k n_k = \bar m$ on $\partial\Omega$. Both are **second-order** PDEs in their respective primary fields ($u_i$ and $\varepsilon^p_e$), so standard $C^0$ (continuous, Lagrange) finite elements suffice for both (unlike Fleck & Hutchinson's original strain gradient theory, which is 4th-order in $u_i$ and needs $C^1$ elements).

The microforce balance says: wherever the moment-stress flux $m_{k,k}$ is nonzero (i.e. $\varepsilon^p_e$ is spatially non-uniform), the *effective* driving stress $q$ departs from the local von Mises stress $\sigma_e$ (this is precisely the mechanism that produces the size effect: near a boundary or interface, plastic strain gradients are steep, $m_{k,k}$ is large, and the material appears to harden more than the local flow stress alone would predict).

## 2. Constitutive laws

**Elastic stress** (unchanged from local plasticity, now using the small-strain elastic strain $\varepsilon^e_{ij}=\varepsilon_{ij}-\varepsilon^p_{ij}$):
$$
\sigma_{ij} = \lambda\,\varepsilon^e_{kk}\,\delta_{ij} + 2\mu\,\varepsilon^e_{ij}.
$$

**Moment stress.** Gudmundson derives $m_k$ from a free energy contribution that penalizes gradients of plastic strain. Taking the simplest (quadratic) form $\psi_g = \tfrac12 \mu L^2 (\varepsilon^p_{e,k}\varepsilon^p_{e,k})$ gives a *linear*, diffusion-like constitutive law
$$
m_k = \mu L^2\, \varepsilon^p_{e,k},
$$
where $L$ is the **single material length scale** of this simplified model (dimension: length) and $\mu$ is the shear modulus (chosen so $L$ has clean units of length; Gudmundson's general form allows an arbitrary monotonic $h(\varepsilon^p_e)$ scaling here, we take the simplest constant-coefficient case). As $L\to 0$ the moment-stress term vanishes and the microforce balance degenerates to $q=\sigma_e$ (the ordinary *local* $J_2$ yield condition). This is the key consistency check we will use once the model is implemented: **$L=0$ (or a sufficiently coarse mesh relative to $L$) must reproduce `src/plasticity.jl` exactly.**

**Flow / hardening law.** For this simplified sub-case, Gudmundson shows the *local* part of the theory (i.e. the relation between $q$ and $\varepsilon^p_e$ used inside the return mapping) reduces to the same isotropic hardening law already implemented:
$$
q = \sigma_y(\varepsilon^p_e) = \sigma_0 + H\varepsilon^p_e + (\sigma_\infty - \sigma_0)\big(1-e^{-\omega \varepsilon^p_e}\big),
$$
exactly `flow_stress(p, alpha)` in `src/plasticity.jl`, with `alpha` $= \varepsilon^p_e$. The crucial difference from local plasticity is **where $q$ comes from**: locally, $q$ was computed pointwise from the trial stress at each quadrature point, independent of neighboring points. Here, $\varepsilon^p_e$ is a genuine **finite element field** (solved for globally via the microforce balance PDE, which couples neighboring points through $m_{k,k}$). The hardening law above still holds pointwise (it is the *material* law relating $q$ to the *local* value of $\varepsilon^p_e$), but $\varepsilon^p_e$ itself can no longer be resolved by an independent per-quadrature-point Newton solve (it has to be solved simultaneously with $u_i$ as a coupled global system, exactly like the mixed (displacement, pressure) system in `MixedElasticity2D.ipynb`).

### Rate-independent yield vs. viscoplastic regularization

The rate-independent version of this model requires **Kuhn-Tucker complementarity** ($\dot\varepsilon^p_e \ge 0$, $q - \sigma_y \le 0$, $\dot\varepsilon^p_e(q-\sigma_y)=0$) to be enforced *simultaneously* with the global microforce balance PDE (i.e. an active-set problem nested inside a coupled Newton solve). The literature on this model class (Fredriksson & Gudmundson and others) explicitly notes this microforce balance system is **numerically stiff** and prone to convergence trouble even without the added complementarity logic.

Gudmundson offers a **viscoplastic regularization** (his Eq. 21) as an alternative that removes the sharp yield surface entirely: plastic flow occurs at *every* stress level, but is vanishingly small below yield, via a smooth power law
$$
\dot\varepsilon^p_e = \dot\varepsilon_0 \left\langle \frac{q}{\sigma_y(\varepsilon^p_e)} \right\rangle^{1/m}, \qquad \dot\varepsilon_0, m \;\text{material parameters},
$$
with $\langle x\rangle = \max(x,0)$. This makes the whole coupled system **smooth everywhere** (no active-set logic), which both sidesteps the known stiffness/convergence issue and lets us reuse Enzyme-based automatic differentiation for the Jacobian exactly as in `src/plasticity.jl`, rather than hand-deriving complementarity-aware tangents. **We adopt the viscoplastic regularization** for the implementation; it converges to the rate-independent limit as $m\to 0$ and $\dot\varepsilon_0$ is chosen small relative to the loading rate.

## 3. Discrete (mixed) weak form

The weak form follows directly from Section 1 (test functions $\delta u_i \to v_i$, $\delta \varepsilon^p_e \to \eta$):
$$
\int_\Omega \sigma_{ij}(u,\varepsilon^p_e)\,v_{i,j}\,dV \;-\; \int_{\Gamma_t} t_i v_i \, dS = 0 \qquad \forall v_i,
$$
$$
\int_\Omega \Big[ m_k(\varepsilon^p_e)\,\eta_{,k} \;-\; \big(\sigma_e(u,\varepsilon^p_e) - q(\varepsilon^p_e)\big)\,\eta \Big]\,dV \;-\; \int_{\Gamma_m} \bar m\,\eta\,dS = 0 \qquad \forall \eta.
$$
(The sign of the $(\sigma_e - q)$ term follows from integrating $m_k\eta_{,k}$ by parts: $\int m_{k,k}\eta = -\int m_k \eta_{,k} + \text{boundary}$, then substituting $m_{k,k}=q-\sigma_e$ from the strong form.) These two residuals are coupled: $\sigma_{ij}$ depends on $\varepsilon^p_e$ through $\varepsilon^p_{ij}=\varepsilon^p_e \cdot 3\sigma'_{ij}/2\sigma_e$, and the second equation depends on $u_i$ through $\sigma_e$.

### Element choice: no inf-sup restriction

In `MixedElasticity2D.ipynb`, the (displacement, pressure) pair is a genuine **saddle-point** (Stokes-like) system: pressure enforces the incompressibility *constraint* $\nabla\cdot u = 0$, and stability requires an inf-sup compatible pairing (e.g. discontinuous pressure one degree lower than displacement).

Here the situation is different. The $(\sigma_e-q)$ term is a **reaction**, not a constraint (the second equation is a reaction-diffusion-type PDE for $\varepsilon^p_e$, not a divergence constraint on $u_i$). There is no LBB/inf-sup condition to satisfy. Consistent with the derivation in Section 1 requiring $H^1$ regularity of $\varepsilon^p_e$ (its gradient enters the weak form), we use:

- **Continuous ($C^0$), equal-order Lagrange elements for both $u_i$ and $\varepsilon^p_e$** (e.g. both tensor-product order $P$, reusing the *same* `FEBasis` machinery already used for `Bu` in the existing code, just with `num_comp=1` for the new field instead of `num_comp=2`).
- The element restriction / connectivity pattern for the scalar field follows the same `FEIndices`-style construction as the continuous-pressure option explored in `MixedElasticity2D.ipynb`, not the discontinuous one.

### Global unknown vector and coupled Jacobian

The global DOF vector becomes $\mathbf U = [\mathbf u;\, \boldsymbol{\varepsilon}^p_e]$ (displacement DOFs followed by scalar plastic-strain-field DOFs, mirroring how `MixedElasticity2D.ipynb` stacks $[\mathbf u;\,\mathbf p]$). The residual is block-structured,
$$
R(\mathbf U) = \begin{bmatrix} R_u(\mathbf u, \boldsymbol\varepsilon^p_e) \\ R_{\varepsilon^p_e}(\mathbf u, \boldsymbol\varepsilon^p_e) \end{bmatrix} = 0,
$$
and the Jacobian has four blocks, all nonzero in general:
$$
J = \begin{bmatrix} \partial R_u/\partial \mathbf u & \partial R_u/\partial \boldsymbol\varepsilon^p_e \\ \partial R_{\varepsilon^p_e}/\partial \mathbf u & \partial R_{\varepsilon^p_e}/\partial \boldsymbol\varepsilon^p_e \end{bmatrix}.
$$
With the viscoplastic regularization (smooth everywhere), the whole block system can be differentiated with Enzyme exactly as `PlasticityJacobian` already does for the local model (no hand-derived tangent operators, no active-set bookkeeping).

## 4. Time integration (load stepping)

As with local plasticity, we use **incremental load stepping**: at each step $n\to n+1$ we know the previous converged state $\varepsilon^p_{e,n}(\mathbf x)$ (a full nodal field now, not just per-quadrature-point history) and solve the coupled residual for $(\mathbf u_{n+1}, \varepsilon^p_{e,n+1})$. The viscoplastic rate law is integrated with a simple backward-Euler step:
$$
\Delta\varepsilon^p_e = \varepsilon^p_{e,n+1} - \varepsilon^p_{e,n} = \Delta t \;\dot\varepsilon_0 \left\langle \frac{q_{n+1}}{\sigma_y(\varepsilon^p_{e,n+1})}\right\rangle^{1/m}.
$$
Unlike local plasticity's `PlasticityState` (a small `struct` per quadrature point, updated by a local Newton solve inside `ReturnMappingDeltaGamma`), history here reduces to a single scalar **nodal field** $\varepsilon^p_{e,n}$ carried between load steps (no per-point tensor history ($C^p_{inv}$) is needed at all, since plane-strain/small-strain plastic strain is now fully determined by the scalar field and the fixed $J_2$ flow direction). This is a further simplification relative to `src/plasticity.jl`.

## Summary before moving to code

| | Local plasticity (`src/plasticity.jl`) | Gradient plasticity (this notebook) |
|---|---|---|
| Kinematics | finite-strain, multiplicative $F=F^eF^p$ | small-strain, additive $\varepsilon=\varepsilon^e+\varepsilon^p$ |
| Plastic strain | tensor, per-quadrature-point history | scalar field $\varepsilon^p_e(\mathbf x)$, genuine FE unknown |
| Unknowns | $\mathbf u$ only | $\mathbf u$ and $\varepsilon^p_e$, coupled |
| Yield/hardening | rate-independent, local Newton return-map | viscoplastic regularization, global coupled Newton |
| History per point | $(\alpha, C^p_{inv})$ | scalar $\varepsilon^p_{e,n}$ nodal value only |
| Length scale | none | single $L$ (via $m_k=\mu L^2\varepsilon^p_{e,k}$) |
| Jacobian | Enzyme (per-element, `PlasticityJacobian`) | Enzyme (per-element, coupled 2x2 block) |

The key limiting-case check for the implementation: **as $L\to 0$, the gradient model must reduce to ordinary local $J_2$ plasticity** (this gives a natural, physically meaningful patch/limit test in addition to the usual FD-Jacobian check).

Next: implement this in `src/gradientplasticity.jl`, reusing `flow_stress`/`hardening_slope` from `src/plasticity.jl` unchanged, and following the same residual/Jacobian/BC conventions as the rest of the package.